# V2 Phase 1 Baseline Analysis

This notebook analyzes artifacts produced by the implementation in `v2/src/`. It does not duplicate dataset, training, evaluation, or plotting logic. No source photographs are embedded.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

working_directory = Path.cwd().resolve()
REPOSITORY_ROOT = next(
    candidate for candidate in (working_directory, *working_directory.parents)
    if (candidate / 'v2').is_dir() and (candidate / 'README.md').is_file()
)
METADATA_PATH = REPOSITORY_ROOT / 'v2/experiments/experiment_001_metadata.json'
HISTORY_PATH = REPOSITORY_ROOT / 'v2/results/baseline_history.json'
METRICS_PATH = REPOSITORY_ROOT / 'v2/results/baseline_test_metrics.json'


## Dataset statistics and class distribution

Counts come from the recorded experiment metadata, which is generated from the frozen local split.

In [ ]:
metadata = json.loads(METADATA_PATH.read_text())
count_frame = pd.DataFrame({
    'train': metadata['dataset']['train_counts'],
    'validation': metadata['dataset']['validation_counts'],
}).fillna(0).astype(int)
display(count_frame)
count_frame.plot.bar(figsize=(9, 5), title='Frozen V1 development split class counts')
plt.ylabel('Images')
plt.tight_layout()
plt.show()


## Training history

The table and curves expose the fixed 20-epoch run without re-running training.

In [ ]:
history = pd.DataFrame(json.loads(HISTORY_PATH.read_text()))
display(history)
display(Image(filename=str(REPOSITORY_ROOT / 'v2/results/baseline_training_curve.png')))
display(Image(filename=str(REPOSITORY_ROOT / 'v2/results/baseline_validation_curve.png')))


## Final evaluation

The test artifact exists only after validation-based checkpoint selection and the registered one-time evaluation.

In [ ]:
metrics = json.loads(METRICS_PATH.read_text())
summary = pd.Series({
    'test_loss': metrics['loss'],
    'test_accuracy': metrics['accuracy'],
    'samples': metrics['samples'],
})
display(summary)
report = pd.DataFrame(metrics['classification_report']).transpose()
display(report)
display(Image(filename=str(REPOSITORY_ROOT / 'v2/results/baseline_confusion_matrix.png')))


## Interpretation boundary

This single fixed run compares modern tooling with the historical CNN concept. It is not a hyperparameter search, does not isolate every numerical difference between TensorFlow and PyTorch, and does not establish food safety.